The Bayesian Network model didn't get great accuracy so I decided to sanity-check by comparing it against a model I was familiar with using.

In [1]:
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer


ds = fetch_ucirepo(id=320)

pp = KBinsDiscretizer(encode="ordinal")
pp.set_output(transform="pandas")

y = pp.fit_transform(ds["data"]["targets"]["G3"].to_frame()).astype(int)
y.value_counts()

X_train, X_test, y_train, y_test = train_test_split(
    ds["data"]["features"],
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, mean_absolute_error


pp = make_column_transformer(
    (OrdinalEncoder(), [
        "school", "sex", "address", "famsize", "Pstatus", "schoolsup", "famsup",
        "paid", "activities", "nursery", "higher", "internet", "romantic",
    ]),
    (OneHotEncoder(), ["Mjob", "Fjob", "reason", "guardian"]),
    ("passthrough", [
        "age", "Medu", "Fedu", "traveltime", "studytime", "failures", "famrel",
        "freetime", "goout", "Dalc", "Walc", "health", "absences",
    ]),
    remainder="drop"
)
X_train_pp = pp.fit_transform(X_train)
X_test_pp = pp.fit_transform(X_test)

model = DecisionTreeRegressor(
    max_depth=4,
    random_state=42,
)
model.fit(X_train_pp, y_train)
y_test_pred = model.predict(X_test_pp)

print("Accuracy:", accuracy_score(y_test, y_test_pred.round()))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred.round()))
print("MAE:", mean_absolute_error(y_test, y_test_pred))

Accuracy: 0.3153846153846154
Confusion matrix:
 [[ 4  2  8  1  0]
 [ 4  3  6  4  0]
 [ 1  5 15 20  0]
 [ 0  2  4 19  0]
 [ 0  4  8 20  0]]
MAE: 0.9664018482652503
